In [1]:
import glob
print(glob.glob("data/**/*.csv", recursive=True))

['data\\02-14-2018.csv', 'data\\02-15-2018.csv', 'data\\02-16-2018.csv', 'data\\02-20-2018.csv', 'data\\02-21-2018.csv', 'data\\02-22-2018.csv', 'data\\02-23-2018.csv', 'data\\02-28-2018.csv', 'data\\03-01-2018.csv', 'data\\03-02-2018.csv']


In [2]:
import pandas as pd, glob

for f in sorted(glob.glob("data/**/*.csv", recursive=True)):
    df = pd.read_csv(f, usecols=["Label"])
    print(f, df["Label"].value_counts().to_dict())

data\02-14-2018.csv {'Benign': 667626, 'FTP-BruteForce': 193360, 'SSH-Bruteforce': 187589}
data\02-15-2018.csv {'Benign': 996077, 'DoS attacks-GoldenEye': 41508, 'DoS attacks-Slowloris': 10990}
data\02-16-2018.csv {'DoS attacks-Hulk': 461912, 'Benign': 446772, 'DoS attacks-SlowHTTPTest': 139890, 'Label': 1}
data\02-20-2018.csv {'Benign': 7372557, 'DDoS attacks-LOIC-HTTP': 576191}
data\02-21-2018.csv {'DDOS attack-HOIC': 686012, 'Benign': 360833, 'DDOS attack-LOIC-UDP': 1730}
data\02-22-2018.csv {'Benign': 1048213, 'Brute Force -Web': 249, 'Brute Force -XSS': 79, 'SQL Injection': 34}
data\02-23-2018.csv {'Benign': 1048009, 'Brute Force -Web': 362, 'Brute Force -XSS': 151, 'SQL Injection': 53}
data\02-28-2018.csv {'Benign': 544200, 'Infilteration': 68871, 'Label': 33}
data\03-01-2018.csv {'Benign': 238037, 'Infilteration': 93063, 'Label': 25}
data\03-02-2018.csv {'Benign': 762384, 'Bot': 286191}


In [3]:
import pandas as pd, numpy as np, glob, os

LABEL_TO_CAT = {
    "Benign": "Benign",
    "DoS attacks-GoldenEye": "Network", "DoS attacks-Slowloris": "Network",
    "DoS attacks-Hulk": "Network", "DoS attacks-SlowHTTPTest": "Network",
    "DDoS attacks-LOIC-HTTP": "Network", "DDOS attack-HOIC": "Network",
    "DDOS attack-LOIC-UDP": "Network",
    "FTP-BruteForce": "Credential", "SSH-Bruteforce": "Credential",
    "Brute Force -Web": "Credential",
    "Brute Force -XSS": "Unmapped", "SQL Injection": "Unmapped",
    "Bot": "Malware", "Infilteration": "Malware",
}

# Fraction of rows to keep per label (small classes kept in full)
KEEP_ALL = ["Brute Force -Web", "Brute Force -XSS", "SQL Injection",
            "DDOS attack-LOIC-UDP", "DoS attacks-Slowloris", "DoS attacks-GoldenEye"]
def keep_frac(label):
    if label == "Benign": return 0.02
    if label in KEEP_ALL: return 1.0
    return 0.10

rng = np.random.default_rng(42)
files = sorted(glob.glob("data/*.csv"))
base_cols = set(pd.read_csv(files[0], nrows=0).columns)

parts = []
for f in files:
    day = os.path.basename(f)[:5]
    kept = 0
    for chunk in pd.read_csv(f, usecols=lambda c: c in base_cols,
                             chunksize=500_000, low_memory=False):
        chunk = chunk[chunk["Label"] != "Label"]          # repeated header rows
        frac = chunk["Label"].map(keep_frac).fillna(0.10)
        chunk = chunk[rng.random(len(chunk)) < frac.values].copy()
        chunk["Day"] = day
        parts.append(chunk)
        kept += len(chunk)
    print(f, "kept", kept)

df = pd.concat(parts, ignore_index=True)
feat = [c for c in df.columns if c not in ("Label", "Timestamp", "Day")]
df[feat] = df[feat].apply(pd.to_numeric, errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feat)
df = df.drop_duplicates()
df["Category"] = df["Label"].map(LABEL_TO_CAT)

print(df.shape)
print(df["Label"].value_counts())
print(df["Category"].value_counts())
df.to_pickle("sample.pkl")

data\02-14-2018.csv kept 51333
data\02-15-2018.csv kept 72315
data\02-16-2018.csv kept 69099
data\02-20-2018.csv kept 205129
data\02-21-2018.csv kept 77583
data\02-22-2018.csv kept 20988
data\02-23-2018.csv kept 21467
data\02-28-2018.csv kept 17987
data\03-01-2018.csv kept 13989
data\03-02-2018.csv kept 43662
(574414, 82)
Label
Benign                      267330
DDOS attack-HOIC             68418
DDoS attacks-LOIC-HTTP       57845
DoS attacks-Hulk             45851
DoS attacks-GoldenEye        41455
Bot                          28516
SSH-Bruteforce               16516
Infilteration                16047
FTP-BruteForce               12268
DoS attacks-Slowloris        10285
DoS attacks-SlowHTTPTest      7225
DDOS attack-LOIC-UDP          1730
Brute Force -Web               611
Brute Force -XSS               230
SQL Injection                   87
Name: count, dtype: int64
Category
Benign        267330
Network       232809
Malware        44563
Credential     29395
Unmapped         317
Name:

In [11]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd, numpy as np, time, pickle, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.svm import SVC

df = pd.read_pickle("sample.pkl")
feat = [c for c in df.columns if c not in ("Label", "Timestamp", "Day", "Category")]
X = df[feat].astype("float32").values
y = (df["Label"] != "Benign").astype(int).values
cats = df["Category"].values

idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.3, stratify=df["Label"], random_state=42)
scaler = StandardScaler().fit(X[idx_train])
X_train = np.clip(scaler.transform(X[idx_train]), -10, 10)
X_test  = np.clip(scaler.transform(X[idx_test]), -10, 10)
y_train, y_test, cats_test = y[idx_train], y[idx_test], cats[idx_test]

COVERAGE_CATS, RECALL_THRESHOLD = ["Network", "Credential", "Malware"], 0.80

def evaluate(pred, name):
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    res = {"tool": name, "f1_score": round(float(f1_score(y_test, pred)), 4),
           "fpr": round(float(fp / (fp + tn)), 4)}
    recall = {c: round(float(pred[cats_test == c].mean()), 3)
              for c in COVERAGE_CATS + ["Unmapped"]}
    res["categories_covered"] = sum(recall[c] >= RECALL_THRESHOLD for c in COVERAGE_CATS)
    res["category_recall"] = recall
    return res

def latency_batch_ms(predict_fn, batch, repeats=5):
    predict_fn(batch)
    t = time.perf_counter()
    for _ in range(repeats):
        predict_fn(batch)
    return round((time.perf_counter() - t) / repeats * 1000, 2)

TIMESTEPS, STEP_FEATS = 6, 13

class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(STEP_FEATS, 64, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        x = x.view(-1, TIMESTEPS, STEP_FEATS)
        _, (h, _) = self.lstm(x)
        return self.head(h[-1]).squeeze(1)

class CNNNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool1d(8))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.2),
                                  nn.Linear(64 * 8, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        return self.head(self.conv(x.unsqueeze(1))).squeeze(1)

def train_model(model, epochs=20, batch=512, patience=3, lr=1e-3):
    Xt = torch.tensor(X_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    n_val = int(0.1 * len(Xt))
    perm = torch.randperm(len(Xt))
    val_i, tr_i = perm[:n_val], perm[n_val:]
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    best, best_state, bad = float("inf"), None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        order = tr_i[torch.randperm(len(tr_i))]
        for i in range(0, len(order), batch):
            b = order[i:i+batch]
            opt.zero_grad()
            loss_fn(model(Xt[b]), yt[b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xt[val_i]), yt[val_i]).item()
        print(f"epoch {ep+1}: val_loss {vl:.4f}")
        if vl < best:
            best, bad = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return round(time.time() - t0)

def make_predict(model):
    model.eval()
    def predict(a):
        with torch.no_grad():
            return (model(torch.tensor(a, dtype=torch.float32)) > 0).numpy().astype(int)
    return predict

print(X_train.shape, X_test.shape)

(402089, 78) (172325, 78)


In [3]:
results = []

torch.manual_seed(42); np.random.seed(42)
lstm = LSTMNet(); lstm_train_s = train_model(lstm)
lstm_predict = make_predict(lstm)
r = evaluate(lstm_predict(X_test), "LSTM-based NIDS")
r["latency_ms"] = latency_batch_ms(lstm_predict, X_test[:1000]); r["train_s"] = lstm_train_s
results.append(r); print(r)

torch.manual_seed(42)
cnn = CNNNet(); cnn_train_s = train_model(cnn)
cnn_predict = make_predict(cnn)
r = evaluate(cnn_predict(X_test), "CNN-based NIDS")
r["latency_ms"] = latency_batch_ms(cnn_predict, X_test[:1000]); r["train_s"] = cnn_train_s
results.append(r); print(r)

def lstm_features(a):
    with torch.no_grad():
        x = torch.tensor(a, dtype=torch.float32).view(-1, TIMESTEPS, STEP_FEATS)
        _, (h, _) = lstm.lstm(x)
    return h[-1].numpy()

sub, _ = train_test_split(np.arange(len(X_train)), train_size=60_000,
                          stratify=y_train, random_state=42)
t0 = time.time()
F_train = lstm_features(X_train[sub])
fscaler = StandardScaler().fit(F_train)
svm = SVC(kernel="rbf", C=1.0, gamma="scale").fit(fscaler.transform(F_train), y_train[sub])
svm_fit_s = round(time.time() - t0)

def hybrid_predict(a):
    return svm.predict(fscaler.transform(lstm_features(a))).astype(int)

r = evaluate(hybrid_predict(X_test), "Hybrid SVM-DL NIDS")
r["latency_ms"] = latency_batch_ms(hybrid_predict, X_test[:1000]); r["train_s"] = lstm_train_s + svm_fit_s
results.append(r); print(r)

torch.save(lstm.state_dict(), "lstm.pt")
torch.save(cnn.state_dict(), "cnn.pt")
pickle.dump({"svm": svm, "fscaler": fscaler, "scaler": scaler}, open("hybrid.pkl", "wb"))
pd.DataFrame(results).to_json("ai_results.json", orient="records", indent=2)
print("saved models and ai_results.json")

epoch 1: val_loss 0.1329
epoch 2: val_loss 0.1119
epoch 3: val_loss 0.1058
epoch 4: val_loss 0.1047
epoch 5: val_loss 0.1026
epoch 6: val_loss 0.1026
epoch 7: val_loss 0.1040
epoch 8: val_loss 0.1000
epoch 9: val_loss 0.0999
epoch 10: val_loss 0.0984
epoch 11: val_loss 0.0980
epoch 12: val_loss 0.0978
epoch 13: val_loss 0.0987
epoch 14: val_loss 0.0977
epoch 15: val_loss 0.0981
epoch 16: val_loss 0.0987
epoch 17: val_loss 0.0970
epoch 18: val_loss 0.0959
epoch 19: val_loss 0.0971
epoch 20: val_loss 0.0985
{'tool': 'LSTM-based NIDS', 'f1_score': 0.9751, 'fpr': 0.0067, 'categories_covered': 2, 'category_recall': {'Network': 0.999, 'Credential': 0.992, 'Malware': 0.715, 'Unmapped': 0.474}, 'latency_ms': 4.95, 'train_s': 156}
epoch 1: val_loss 0.1417
epoch 2: val_loss 0.1159
epoch 3: val_loss 0.1047
epoch 4: val_loss 0.1030
epoch 5: val_loss 0.0985
epoch 6: val_loss 0.0984
epoch 7: val_loss 0.0993
epoch 8: val_loss 0.0985
epoch 9: val_loss 0.0949
epoch 10: val_loss 0.0932
epoch 11: val_los

In [ ]:
import json, numpy as np

lat = {}
for name, fn in [("LSTM-based NIDS", lstm_predict), ("CNN-based NIDS", cnn_predict),
                 ("Hybrid SVM-DL NIDS", hybrid_predict)]:
    lat[name] = float(np.median([latency_batch_ms(fn, X_test[:1000], repeats=20) for _ in range(3)]))
print(lat)

ai = {r["tool"]: r for r in json.load(open("ai_results.json"))}
NOTE = ("Measured on a stratified CSE-CIC-IDS2018 sample (574,414 rows, 70/30 stratified split, seed 42, CPU only). "
        "Latency = ms per batch of 1,000 flows (median of 3 timings). Coverage = categories with >=80% recall "
        "out of the 3 measurable; reconnaissance has no ground truth in this dataset.")
tools = []
for name in ["LSTM-based NIDS", "CNN-based NIDS", "Hybrid SVM-DL NIDS"]:
    r = ai[name]
    tools.append({"name": name, "f1_score": r["f1_score"], "fpr": r["fpr"],
                  "latency_ms": round(lat[name], 2), "categories_covered": r["categories_covered"],
                  "total_categories": 4,
                  "notes": NOTE + f" Training time {r['train_s']} s. Category recall: {r['category_recall']}"})
tools.append({"name": "Snort (rule-based NIDS)", "f1_score": 0.0, "fpr": 0.0, "latency_ms": 4.4,
              "categories_covered": 0, "total_categories": 4,
              "notes": "Snort 2.9.20-WIN64, community rules (561 active), checksums off (-k none), HOME_NET any. "
                       "Run on victim-host captures for 02-14 (4.72M packets) and 02-16 (18.9M packets): 0 alerts. "
                       "Latency = ms per 1,000 packets, including start-up. FPR of 0 reflects no alerts at all."})
json.dump(tools, open("tools_data.json", "w"), indent=2)
print("written")

{'LSTM-based NIDS': 2.28, 'CNN-based NIDS': 14.63, 'Hybrid SVM-DL NIDS': 274.14}
written


: 